In [10]:
!kaggle datasets download -d rohiteng/amazon-sales-dataset

Dataset URL: https://www.kaggle.com/datasets/rohiteng/amazon-sales-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100%|██████████████████████████████████████| 3.85M/3.85M [00:01<00:00, 4.16MB/s]
100%|██████████████████████████████████████| 3.85M/3.85M [00:01<00:00, 3.65MB/s]


## Amazon Sales Dataset - ETL Process

### 📊 분석 배경

> "현재 신규고객이 점점 줄어들고 있습니다. 
> 하지만 지금 저희는 acquisition보다는 retention에 더 집중해야 할 때입니다. 
> 재결제자의 비율을 늘릴 방안을 마련해보세요"

재구매율을 높이기 위한 분석을 위해 Kaggle의 Amazon Sales Dataset을 활용한다.

---

### 🔄 ETL Process

#### 1. Extract (추출)
- **데이터 소스**: Kaggle - Amazon Sales Dataset
- **링크**: https://www.kaggle.com/datasets/rohiteng/amazon-sales-dataset
- **데이터 크기**: 100,000건
- **기간**: 2020-01-01 ~ 2024-12-31

#### 2. Transform (변환)
- **컬럼 선택**: 분석에 필요한 8개 컬럼만 추출
  - OrderDate, CustomerID, ProductID, Category, Brand, Discount, TotalAmount, PaymentMethod
- **데이터 정제**:
  - 결측치 처리
  - 데이터 타입 변환 (OrderDate → datetime)
  - 불필요한 컬럼 제거

#### 3. Load (적재)
- **데이터베이스**: MySQL (localhost)
- **테이블명**: orders_info
- **적재 완료**: 100,000건

본 분석에서는 Kaggle에서 제공하는 **Amazon Sales Dataset** 데이터를 활용하였다.  
링크: https://www.kaggle.com/datasets/rohiteng/amazon-sales-dataset  
분석을 수행하기 위해 다음과 같은 ETL 과정을 진행하였다.

### 1. Extract
- Kaggle에서 데이터를 다운로드하였다.
- CSV 파일을 Python 환경에서 불러왔다.

In [1]:
import pandas as pd

df = pd.read_csv("../Amazon.csv", encoding="utf-8-sig")
df.head()

,OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
0,ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,106.59,0.00,0.00,0.09,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
1,ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,251.37,0.05,19.10,1.74,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
2,ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,35.03,0.10,7.57,5.91,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
3,ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,33.58,0.15,11.42,5.53,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
4,ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,515.64,0.25,38.67,9.23,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411


In [2]:
df.shape

(100000, 20)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 20 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   OrderID        100000 non-null  object 
 1   OrderDate      100000 non-null  object 
 2   CustomerID     100000 non-null  object 
 3   CustomerName   100000 non-null  object 
 4   ProductID      100000 non-null  object 
 5   ProductName    100000 non-null  object 
 6   Category       100000 non-null  object 
 7   Brand          100000 non-null  object 
 8   Quantity       100000 non-null  int64  
 9   UnitPrice      100000 non-null  float64
 10  Discount       100000 non-null  float64
 11  Tax            100000 non-null  float64
 12  ShippingCost   100000 non-null  float64
 13  TotalAmount    100000 non-null  float64
 14  PaymentMethod  100000 non-null  object 
 15  OrderStatus    100000 non-null  object 
 16  City           100000 non-null  object 
 17  State          100000 non-null

### 📋 컬럼 별 정보

* **OrderID**: 각 주문을 식별하기 위한 고유 주문 번호
* **OrderDate**: 주문이 발생한 날짜
* **CustomerID**: 고객을 식별하기 위한 고유 ID
* CustomerName: 고객 이름
* **ProductID**: 상품을 식별하기 위한 고유 ID
* ProductName: 판매된 상품 이름
* **Category**: 상품의 카테고리 (상품 유형)
* **Brand**: 상품의 브랜드 또는 제조사 (예: Samsung, Apple, Puma, Lenovo 등)
* Quantity: 주문된 상품의 수량
* UnitPrice: 할인 적용 전 상품 1개당 판매 가격
* **Discount**: 주문에 적용된 할인 금액 또는 할인율
* Tax: 주문에 적용된 세금 금액
* ShippingCost: 상품 배송에 발생한 비용
* **TotalAmount**: 주문의 최종 결제 금액 (할인, 세금, 배송비 등이 반영된 금액)
* **PaymentMethod**: 결제 방식 (예: Credit Card, Debit Card, PayPal 등)
* OrderStatus: 주문 상태 (예: Delivered, Shipped, Cancelled 등)
* City: 주문이 발생한 도시
* State: 주문이 발생한 주(State)
* Country: 주문이 발생한 국가
* SellerID: 상품을 판매한 판매자 또는 공급업체 ID

### 2. Transform
- 불필요한 컬럼 및 결측 데이터를 제거하였다.
- 날짜 데이터를 datetime 형식으로 변환하였다.
- 분석에 필요한 형태로 데이터를 정제하였다.

In [4]:
columns_need = ['OrderID', 'OrderDate', 'CustomerID', 'ProductID', 'Category', 'Brand', 'Discount', 'TotalAmount', 'PaymentMethod']
amazon_retension = df[columns_need].copy()

In [5]:
amazon_retension.head()

,OrderID,OrderDate,CustomerID,ProductID,Category,Brand,Discount,TotalAmount,PaymentMethod
0,ORD0000001,2023-01-31,CUST001504,P00014,Books,BrightLux,0.00,319.86,Debit Card
1,ORD0000002,2023-12-30,CUST000178,P00040,Home & Kitchen,UrbanStyle,0.05,259.64,Amazon Pay
2,ORD0000003,2022-05-10,CUST047516,P00044,Clothing,UrbanStyle,0.10,108.06,Debit Card
3,ORD0000004,2023-07-18,CUST030059,P00041,Home & Kitchen,Zenith,0.15,159.66,Cash on Delivery
4,ORD0000005,2023-02-04,CUST048677,P00029,Clothing,KiddoFun,0.25,821.36,Credit Card


In [6]:
amazon_retension['OrderDate'] = pd.to_datetime(amazon_retension['OrderDate'], format="%Y-%m-%d")

In [7]:
amazon_retension.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   OrderID        100000 non-null  object        
 1   OrderDate      100000 non-null  datetime64[ns]
 2   CustomerID     100000 non-null  object        
 3   ProductID      100000 non-null  object        
 4   Category       100000 non-null  object        
 5   Brand          100000 non-null  object        
 6   Discount       100000 non-null  float64       
 7   TotalAmount    100000 non-null  float64       
 8   PaymentMethod  100000 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(6)
memory usage: 6.9+ MB


### 3. Load

정제된 데이터를 MySQL 데이터베이스에 적재하였다.

- MySQL에서 `ecommerce` 데이터베이스를 생성하였다.
- SQLAlchemy를 활용하여 MySQL 서버와 Python 환경을 연결하였다.
- pandas의 `to_sql()` 함수를 이용하여 정제된 데이터를 `orders` 테이블로 적재하였다.

In [8]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()  # .env 파일 로드

# MySQL 연결
engine = create_engine(
    f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

# 데이터 적재
amazon_retension.to_sql(
    name='orders_info',
    con=engine,
    if_exists="replace",
    index=False
)

100000